# 07. EDA와 전처리

이번 노트북에서는 심장 질환 예측 데이터 `heart.csv`를 사용해 EDA와 전처리를 연습합니다.

아직 모델을 학습하지 않습니다. 모델 전에 데이터를 보는 습관을 만드는 것이 목표입니다.

In [1]:
import pandas as pd  # 표 형태 데이터를 다루는 라이브러리입니다.
from sklearn.model_selection import train_test_split  # 데이터를 학습용/검증용으로 나누는 도구입니다.
from sklearn.preprocessing import StandardScaler  # 숫자 컬럼의 스케일을 맞추는 도구입니다.

DATA_PATH = r"C:\work\dataset\diabetes_or_cardiovascular\heart.csv" 

## 1. 데이터 불러오기

`pd.read_csv()`는 CSV 파일을 읽어서 DataFrame으로 만들어줍니다.

DataFrame은 엑셀 표와 비슷한 pandas의 데이터 구조입니다.

In [2]:
df = pd.read_csv(DATA_PATH)  # CSV 파일을 읽습니다.

print("데이터 앞 5줄:")
display(df.head())  # head()는 앞부분 5줄을 보여줍니다.

데이터 앞 5줄:


,Age,Sex,ChestPainType,RestingBP,Cholesterol,FastingBS,RestingECG,MaxHR,ExerciseAngina,Oldpeak,ST_Slope,HeartDisease
0,40,M,ATA,140,289,0,Normal,172,N,0.0,Up,0
1,49,F,NAP,160,180,0,Normal,156,N,1.0,Flat,1
2,37,M,ATA,130,283,0,ST,98,N,0.0,Up,0
3,48,F,ASY,138,214,0,Normal,108,Y,1.5,Flat,1
4,54,M,NAP,150,195,0,Normal,122,N,0.0,Up,0


## 2. 데이터 크기와 컬럼 확인하기

In [3]:
print("데이터 모양:", df.shape)  # (행 개수, 열 개수)를 보여줍니다.
print("컬럼 목록:")
print(df.columns.tolist())  # 컬럼 이름들을 리스트로 보여줍니다.

데이터 모양: (918, 12)
컬럼 목록:
['Age', 'Sex', 'ChestPainType', 'RestingBP', 'Cholesterol', 'FastingBS', 'RestingECG', 'MaxHR', 'ExerciseAngina', 'Oldpeak', 'ST_Slope', 'HeartDisease']


## 3. 컬럼 타입과 결측치 확인하기

`info()`는 컬럼 타입과 비어 있지 않은 값의 개수를 보여줍니다.

In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 918 entries, 0 to 917
Data columns (total 12 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   Age             918 non-null    int64  
 1   Sex             918 non-null    object 
 2   ChestPainType   918 non-null    object 
 3   RestingBP       918 non-null    int64  
 4   Cholesterol     918 non-null    int64  
 5   FastingBS       918 non-null    int64  
 6   RestingECG      918 non-null    object 
 7   MaxHR           918 non-null    int64  
 8   ExerciseAngina  918 non-null    object 
 9   Oldpeak         918 non-null    float64
 10  ST_Slope        918 non-null    object 
 11  HeartDisease    918 non-null    int64  
dtypes: float64(1), int64(6), object(5)
memory usage: 86.2+ KB


In [5]:
# isnull()은 값이 비어 있으면 True, 아니면 False를 반환합니다.
# sum()을 붙이면 컬럼별 결측치 개수를 셀 수 있습니다.
missing_counts = df.isnull().sum()

print("컬럼별 결측치 개수:")
print(missing_counts)

컬럼별 결측치 개수:
Age               0
Sex               0
ChestPainType     0
RestingBP         0
Cholesterol       0
FastingBS         0
RestingECG        0
MaxHR             0
ExerciseAngina    0
Oldpeak           0
ST_Slope          0
HeartDisease      0
dtype: int64


## 4. 정답 컬럼 확인하기

이 데이터에서 정답은 `HeartDisease`입니다.

0은 심장 질환 없음, 1은 심장 질환 있음을 의미합니다.

In [6]:
target_col = "HeartDisease"

print(df[target_col].value_counts())  # 정답값별 개수를 확인합니다.
print(df[target_col].value_counts(normalize=True))  # 비율로도 확인합니다.

HeartDisease
1    508
0    410
Name: count, dtype: int64
HeartDisease
1    0.553377
0    0.446623
Name: proportion, dtype: float64


## 5. 숫자 컬럼과 문자 컬럼 나누기

모델은 문자를 그대로 이해하지 못하므로 문자 컬럼은 숫자로 바꿔야 합니다.

In [7]:
# select_dtypes(include="number")는 숫자 컬럼만 고릅니다.
numeric_cols = df.drop(columns=[target_col]).select_dtypes(include="number").columns.tolist()

# select_dtypes(exclude="number")는 숫자가 아닌 컬럼을 고릅니다.
categorical_cols = df.drop(columns=[target_col]).select_dtypes(exclude="number").columns.tolist()

print("숫자 컬럼:", numeric_cols)
print("문자 컬럼:", categorical_cols)

숫자 컬럼: ['Age', 'RestingBP', 'Cholesterol', 'FastingBS', 'MaxHR', 'Oldpeak']
문자 컬럼: ['Sex', 'ChestPainType', 'RestingECG', 'ExerciseAngina', 'ST_Slope']


## 6. 입력 X와 정답 y 분리하기

In [8]:
X = df.drop(columns=[target_col])  # 정답 컬럼을 제외한 나머지가 입력입니다.
y = df[target_col]  # 정답 컬럼입니다.

print("X 모양:", X.shape)
print("y 모양:", y.shape)

X 모양: (918, 11)
y 모양: (918,)


## 7. 문자 컬럼을 숫자로 바꾸기

`pd.get_dummies()`는 문자 컬럼을 원핫 인코딩해줍니다.

In [9]:
X_encoded = pd.get_dummies(X, columns=categorical_cols, drop_first=False)

print("인코딩 전 X 모양:", X.shape)
print("인코딩 후 X 모양:", X_encoded.shape)
display(X_encoded.head())

인코딩 전 X 모양: (918, 11)
인코딩 후 X 모양: (918, 20)


,Age,RestingBP,Cholesterol,FastingBS,MaxHR,Oldpeak,Sex_F,Sex_M,ChestPainType_ASY,ChestPainType_ATA,ChestPainType_NAP,ChestPainType_TA,RestingECG_LVH,RestingECG_Normal,RestingECG_ST,ExerciseAngina_N,ExerciseAngina_Y,ST_Slope_Down,ST_Slope_Flat,ST_Slope_Up
0,40,140,289,0,172,0.0,False,True,False,True,False,False,False,True,False,True,False,False,False,True
1,49,160,180,0,156,1.0,True,False,False,False,True,False,False,True,False,True,False,False,True,False
2,37,130,283,0,98,0.0,False,True,False,True,False,False,False,False,True,True,False,False,False,True
3,48,138,214,0,108,1.5,True,False,True,False,False,False,False,True,False,False,True,False,True,False
4,54,150,195,0,122,0.0,False,True,False,False,True,False,False,True,False,True,False,False,False,True


## 8. 학습용/검증용 데이터 나누기

`train_test_split()`은 데이터를 학습용과 검증용으로 나눕니다.

`stratify=y`는 정답 0과 1의 비율이 양쪽에 비슷하게 유지되도록 돕습니다.

In [10]:
X_train, X_val, y_train, y_val = train_test_split(
    X_encoded,
    y,
    test_size=0.2,  # 전체 데이터의 20%를 검증용으로 사용합니다.
    random_state=42,  # 결과를 재현하기 위해 랜덤 기준을 고정합니다.
    stratify=y  # 정답 비율을 비슷하게 유지합니다.
)

print("X_train:", X_train.shape)
print("X_val:", X_val.shape)
print("y_train:", y_train.shape)
print("y_val:", y_val.shape)

X_train: (734, 20)
X_val: (184, 20)
y_train: (734,)
y_val: (184,)


## 9. 스케일링

숫자 범위를 비슷하게 맞추기 위해 `StandardScaler`를 사용합니다.

중요: scaler는 학습 데이터에만 `fit`합니다. 검증 데이터에는 `transform`만 합니다.

In [11]:
scaler = StandardScaler()

# fit_transform()은 학습 데이터에서 평균/표준편차를 배운 뒤 변환까지 합니다.
X_train_scaled = scaler.fit_transform(X_train)

# transform()은 학습 데이터에서 배운 기준으로 검증 데이터를 변환만 합니다.
X_val_scaled = scaler.transform(X_val)

print("스케일링 후 X_train 모양:", X_train_scaled.shape)
print("스케일링 후 X_val 모양:", X_val_scaled.shape)
print("스케일링 후 첫 번째 행 일부:", X_train_scaled[0][:5])

스케일링 후 X_train 모양: (734, 20)
스케일링 후 X_val 모양: (184, 20)
스케일링 후 첫 번째 행 일부: [ 0.9700116   0.3390158   0.12713661  1.83549656 -0.32451998]


## 정리

이번 장에서는 모델 전에 해야 할 일을 배웠습니다.

```text
데이터 불러오기
크기/컬럼/결측치 확인
정답 분포 확인
X와 y 분리
문자 컬럼 인코딩
학습/검증 분리
스케일링
```

다음 장부터는 이 전처리된 데이터를 실제 모델에 넣을 수 있습니다.